# 과제 10 · 04 · EventCard 로 질의를 만들고 하이브리드로 찾기

## 현업 시나리오
근거를 찾으려면 질의가 필요합니다. 그런데 EventCard 는 문장이 아니라 숫자와 코드입니다. 카드에서 질의를 만들어야 하고, 매뉴얼에는 부품 번호(`PN-CF-3105`)처럼 글자가 정확히 맞아야 찾는 것과 "온도가 안 떨어진다"처럼 뜻이 통해야 찾는 것이 함께 있습니다.

## 학습 목표
- EventCard 의 후보 유형과 근거 신호로 질의를 만든다. LLM 에게 질의를 쓰게 하지 않는다.
- 글자 검색(BM25)과 의미 검색이 서로 다른 것을 찾는 것을 본다.
- `EnsembleRetriever` 로 두 결과를 합쳐 한쪽만으로 놓치는 것을 함께 잡는다.

## 직접 조립
완성된 App을 가져오지 않습니다. 아래에서 작은 fixture와 핵심 함수를 직접 만듭니다.

### 1단계 · 카드에서 질의 만들기

In [ ]:
# 후보 유형마다 "판정 기준과 점검 절차" 를 찾는 질의 하나, 그리고 확신도 대응 질의 하나.
# 근거 신호(모델이 가장 크게 본 센서) 두 개를 덧붙여 무엇이 이상했는지를 싣습니다.
# 관찰 포인트: 같은 카드는 언제나 같은 질의가 됩니다. 그래야 결과를 비교하고 녹화할 수 있습니다.
practice_card = {
    "event_id": "EVT-2025-0002", "severity": "alarm",
    "prediction": {"candidates": ["HDF", "PWF"], "confidence_level": "medium",
                   "top_signals": [{"feature": "temp_diff_k"}, {"feature": "rotational_speed_rpm"},
                                   {"feature": "temp_diff_k"}]},
}
PRACTICE_TYPE_NAMES = {"TWF": "공구 마모 고장(TWF)", "HDF": "열 방산 고장(HDF)",
                       "PWF": "전력 고장(PWF)", "OSF": "과부하 고장(OSF)"}
PRACTICE_SIGNAL_NAMES = {"temp_diff_k": "온도 차", "rotational_speed_rpm": "회전수", "torque_nm": "토크"}

def practice_queries(card):
    p = card["prediction"]
    signals = list(dict.fromkeys(PRACTICE_SIGNAL_NAMES[s["feature"]] for s in p["top_signals"]))
    hint = ", ".join(signals[:2])
    queries = [(f"{t} 판정·점검", f"{PRACTICE_TYPE_NAMES[t]} 판정 기준과 점검 절차, 조치 — {hint}")
               for t in p["candidates"]]
    queries.append(("확신도 대응", f"확신도 {p['confidence_level']} {card['severity']} 이벤트 대응 방법"))
    return queries

for practice_label, practice_text in practice_queries(practice_card):
    print(f"  {practice_label:10} {practice_text}")

**결과 해설** — 후보 유형이 둘(HDF·PWF)이라 유형 질의 둘과 확신도 질의 하나, 모두 셋이 나왔습니다. 근거 신호는 "온도 차"가 두 번 있었지만 한 번만 들어갑니다. 질의는 코드가 만들므로 같은 카드는 언제나 같은 질의가 됩니다.

### 2단계 · 두 검색기는 서로 다른 것을 찾는다

In [ ]:
# 글자 검색은 BM25, 의미 검색은 작은 "개념 사전" 임베딩으로 만듭니다.
# 개념 사전은 진짜 의미 모델이 아닙니다. "온도가"·"냉각" 을 같은 '열' 개념으로 묶어 뜻을 흉내 냅니다.
# 관찰 포인트: 부품 번호 질의는 글자 검색만, 상황 질의는 의미 검색만 맞힙니다. 각자 1위 하나씩 봅니다.
import re
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.vectorstores import InMemoryVectorStore

# 절 단위로 자르고 머리말 경로를 붙인 매뉴얼 조각입니다(Notebook 02 의 결과).
PRACTICE_SECTIONS = [
    ("MC01-MM 4.1.2", "[정비 매뉴얼 > 4.1 공구 마모 고장 (TWF) > 4.1.2 판정 기준] 공구 마모 시간이 200분 이상이면 위험 구간이다. 240분을 넘기면 사용 한계다."),
    ("MC01-MM 4.1.3", "[정비 매뉴얼 > 4.1 공구 마모 고장 (TWF) > 4.1.3 점검 절차] 날 끝 마모 폭을 측정한다. 마모 폭이 0.3mm 를 넘으면 공구(PN-TL-1008)를 교체한다."),
    ("MC01-MM 4.2.2", "[정비 매뉴얼 > 4.2 열 방산 고장 (HDF) > 4.2.2 판정 기준] 공정 온도와 공기 온도의 차이가 8.6 K 미만이고 회전수가 1,380 rpm 미만이면 해당한다."),
    ("MC01-MM 4.2.3", "[정비 매뉴얼 > 4.2 열 방산 고장 (HDF) > 4.2.3 점검 절차] 흡기 필터(PN-FL-3120)의 압력차 표시를 확인한다. 냉각팬 회전을 확인한다. 방열판 먼지를 제거한다."),
    ("MC01-MM 4.2.4", "[정비 매뉴얼 > 4.2 열 방산 고장 (HDF) > 4.2.4 권장 조치] 필터 압력차가 기준을 넘으면 흡기 필터를 교체한다. 냉각팬이 멈췄으면 냉각팬 모듈(PN-CF-3105)을 교체한다."),
    ("SOP-EA-01 8", "[이상 이벤트 대응 절차 > 8. 오탐 처리] 판정 기준에 해당하지 않는 경고는 육안·청음 점검 후 이상이 없으면 오탐 점검 기록을 남긴다."),
]

PRACTICE_CONCEPTS = {"열": ["온도", "냉각", "방열", "바람"], "필터": ["필터", "흡기", "압력차"],
                     "공구": ["공구", "마모", "날"], "기준": ["기준", "미만", "이상"]}

class PracticeConceptEmbeddings(Embeddings):
    def _vector(self, text):
        return [1.0] + [float(sum(text.count(w) for w in words)) for words in PRACTICE_CONCEPTS.values()]
    def embed_documents(self, texts):
        return [self._vector(t) for t in texts]
    def embed_query(self, text):
        return self._vector(text)

def practice_words(text):
    return re.findall(r"[0-9A-Za-z가-힣]+", text)

practice_documents = [Document(page_content=text, metadata={"citation": cid}) for cid, text in PRACTICE_SECTIONS]
practice_keyword = BM25Retriever.from_documents(practice_documents, preprocess_func=practice_words, k=1)
practice_dense = InMemoryVectorStore.from_documents(
    practice_documents, PracticeConceptEmbeddings()).as_retriever(search_kwargs={"k": 1})

practice_code_query = "PN-CF-3105 교체"
practice_situation_query = "온도가 잘 안 떨어지고 바람이 약해요"
practice_hits = {}
for practice_name, practice_q in (("부품 번호", practice_code_query), ("상황", practice_situation_query)):
    practice_hits[practice_name] = {
        "글자": [d.metadata["citation"] for d in practice_keyword.invoke(practice_q)],
        "의미": [d.metadata["citation"] for d in practice_dense.invoke(practice_q)]}
    print(f"{practice_name:6} 글자 검색 {practice_hits[practice_name]['글자']} · 의미 검색 {practice_hits[practice_name]['의미']}")

**결과 해설** — 부품 번호 질의는 BM25 가 `PN-CF-3105` 가 적힌 4.2.4(권장 조치)를 찾고, 개념 사전 임베딩은 이 번호의 뜻을 몰라 오탐 처리 절(SOP-EA-01 8)을 냅니다. 상황 질의는 반대입니다. "온도가"·"바람이"는 매뉴얼에 그 글자 그대로 없어 BM25 점수가 모두 0인데도 **BM25 는 무언가를 1위로 돌려줍니다**(SOP-EA-01 8). 의미 검색은 '열' 개념으로 HDF 점검 절차(4.2.3)를 찾습니다.

### 3단계 · EnsembleRetriever 로 합치기

In [ ]:
# EnsembleRetriever 가 두 순위를 RRF(Reciprocal Rank Fusion)로 합칩니다. 융합식을 직접 쓰지 않습니다.
# 앱과 같이 가중치는 0.5 / 0.5 로 둡니다.
# 관찰 포인트: 합친 결과에는 양쪽이 찾은 것이 모두 들어옵니다. 한쪽만 쓰면 둘 중 하나를 잃습니다.
from langchain_classic.retrievers import EnsembleRetriever

practice_hybrid = EnsembleRetriever(retrievers=[practice_dense, practice_keyword], weights=[0.5, 0.5])
practice_fused = {}
for practice_name, practice_q in (("부품 번호", practice_code_query), ("상황", practice_situation_query)):
    practice_fused[practice_name] = [d.metadata["citation"] for d in practice_hybrid.invoke(practice_q)]
    print(f"{practice_name:6} 융합 {practice_fused[practice_name]}")

assert practice_queries(practice_card)[0][1].endswith("온도 차, 회전수")
assert practice_hits["부품 번호"]["글자"] == ["MC01-MM 4.2.4"], "글자 검색이 부품 번호를 못 찾았습니다"
assert practice_hits["부품 번호"]["의미"] != ["MC01-MM 4.2.4"], "의미 검색만으로 부품 번호를 찾았습니다"
assert not practice_hits["상황"]["글자"][0].startswith("MC01-MM 4.2"), "글자 검색만으로 상황 질의를 맞혔습니다"
assert practice_hits["상황"]["의미"][0].startswith("MC01-MM 4.2")
assert "MC01-MM 4.2.4" in practice_fused["부품 번호"]
assert any(c.startswith("MC01-MM 4.2") for c in practice_fused["상황"])

**결과 해설** — 두 질의 모두 합친 결과 두 개 중 하나가 정답 절입니다. 부품 번호 질의에서는 BM25 의 4.2.4 가, 상황 질의에서는 의미 검색의 4.2.3 이 살아남습니다. 융합은 한쪽 결과를 버리지 않고 순위만 다시 매깁니다. 대신 틀린 쪽의 결과(SOP-EA-01 8)도 함께 들어오므로, 근거를 쓰는 쪽(초안과 검증)이 그것을 걸러야 합니다.

## 직접 해 볼 과제
가중치를 `[0.9, 0.1]`(의미 검색 위주)과 `[0.1, 0.9]`(글자 검색 위주)로 바꿔 보세요. 두 질의의 융합 1위가 어떻게 바뀝니까? 실제 앱은 왜 0.5 / 0.5 로 두었을까요?

In [ ]:
# 가중치만 바꿉니다. 검색기 둘과 질의는 그대로입니다.
# 1위가 어느 쪽 검색기의 1위를 따라가는지 보세요.
practice_try_weights = [0.9, 0.1]
practice_try = EnsembleRetriever(retrievers=[practice_dense, practice_keyword], weights=practice_try_weights)
for practice_name, practice_q in (("부품 번호", practice_code_query), ("상황", practice_situation_query)):
    print(practice_name, [d.metadata["citation"] for d in practice_try.invoke(practice_q)])

## 중간 결과
카드 하나에서 나온 질의 셋, 두 질의에 대한 글자 검색과 의미 검색의 결과, 그리고 융합 결과에 양쪽의 정답이 모두 들어오는 것을 확인합니다.

## 실패 경계
여기 의미 검색은 개념 사전으로 뜻을 흉내 낸 것이라 **순위 자체는 대본입니다.** 실제 앱은 녹화된 OpenAI 임베딩을 쓰므로 순위가 진짜입니다. 그래도 하이브리드가 모든 것을 찾지는 못합니다. 실제 카드 111장의 유형 질의 126개 중 1개(EVT-2025-0044 HDF)는 근거 신호가 예측 유형의 것뿐이라 유형 절을 놓칩니다. 그래서 앱은 판정 기준 절을 검색에 맡기지 않고 절 번호로 바로 붙입니다.

## 실제 app 연결
과제 10 App 의 `evidence.py::card_queries` 가 같은 규칙으로 질의를 만들고, `evidence.py::hybrid` 가 녹화된 임베딩의 FAISS 와 `shared/rag/retrievers.py::keyword_retriever`(한국어 2글자 묶음 BM25)를 0.5 / 0.5 로 합칩니다. `evidence.py::find_evidence` 가 질의마다 3개씩 찾고 같은 절은 한 번만 남깁니다.

## 다음 Notebook 연결
다음 `05_history_tool.ipynb` 에서는 매뉴얼이 아니라 정비 이력을 조회하는 Tool 을 만듭니다.